# 🧠 Continual Learning: From Beginner to Research Level

**A Complete Interactive Textbook with Python Implementations**

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ha0wan9/NMA-NeuroAI-Online-Learning-Project/blob/main/playground/continual_learning_complete.ipynb)

---

## Table of Contents

| Part | Topic |
|------|-------|
| **Part 1** | Introduction to Continual Learning |
| **Part 2** | Types of Continual Learning |
| **Part 3** | Continual Learning Methods (All Families) |
| **Part 4** | Dataset Preparation (MNIST Benchmarks) |
| **Part 5** | Evaluation Metrics |
| **Part 6** | Experiments & Comparisons |
| **Part 7** | Visualization Suite |
| **Part 8** | Frequently Asked Questions |
| **Part 9** | Mini Research Section – Predictive Coding & Future Directions |

---

> **Prerequisites**: Basic Python, NumPy, and familiarity with neural networks.  
> **Estimated Time**: 8–20 hours (suitable as a semester-long resource).


## 🔧 Environment Setup & Imports

In [ ]:
# ============================================================
# CELL 0: Environment Setup
# Uncomment the line below if running on Colab / fresh environment
# !pip install torch torchvision matplotlib numpy scikit-learn tqdm seaborn
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

import copy, time, random, warnings
from abc import ABC, abstractmethod
from collections import defaultdict
from typing import List, Dict, Tuple, Optional
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ── Reproducibility ─────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

# ── Device ──────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅  Device : {DEVICE}')
print(f'✅  PyTorch: {torch.__version__}')

# ── Global plot style (dark theme) ─────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9', 'xtick.color': '#8b949e',
    'ytick.color': '#8b949e', 'grid.color': '#21262d',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'legend.facecolor': '#161b22', 'legend.edgecolor': '#30363d',
})

# ── Shared colour palette ───────────────────────────────────
COLORS = {
    'naive': '#ef4444', 'replay': '#3b82f6', 'ewc': '#10b981',
    'si': '#f59e0b', 'lwf': '#8b5cf6', 'gem': '#ec4899',
    'agem': '#06b6d4', 'progressive': '#84cc16',
    'tasks': ['#f472b6','#60a5fa','#34d399','#fbbf24','#a78bfa',
              '#fb923c','#22d3ee','#86efac','#fde68a','#c4b5fd'],
}
print('✅  All imports successful.')


---
# PART 1 : Introduction to Continual Learning
---


## 1.1  What is Continual Learning?

**Continual Learning** (also called *lifelong learning* or *incremental learning*) is the ability of
a learning system to **acquire new knowledge over time without forgetting previously learned information**.

The central tension is the **stability–plasticity dilemma**:

$$
\text{Plasticity} \leftrightarrow \text{Stability}
$$

- **Plasticity** – ability to learn *new* information.
- **Stability** – ability to *retain* old information.

A system that is too plastic forgets old tasks; a system that is too stable cannot learn new ones.

### Formal objective

$$
\boxed{\max_\theta \sum_{t=1}^{T} \mathcal{L}_t(\theta)}
\quad \text{where tasks } \mathcal{T}_1, \mathcal{T}_2, \ldots, \mathcal{T}_T \text{ arrive sequentially}
$$

| Property | Standard DNN | Ideal CL System | Brain |
|----------|-------------|-----------------|-------|
| Sequential learning | ❌ Catastrophic forgetting | ✅ | ✅ |
| Memory efficiency | ❌ Grows with tasks | ✅ Bounded | ✅ |
| Forward transfer | ❌ Minimal | ✅ | ✅ |
| Backward transfer | ❌ Negative | ✅ | Mostly ✅ |


## 1.2  Why Traditional Supervised Learning Fails

Traditional supervised learning assumes:
1. **i.i.d. data** – all samples drawn from the same fixed distribution.
2. **Static distribution** – $p(x,y)$ does not change over time.
3. **Full data access** – all training data available simultaneously.

In real deployments all three assumptions are violated.

### Catastrophic Forgetting / Interference

When trained sequentially $\mathcal{T}_1 \to \mathcal{T}_2$, gradient updates for $\mathcal{T}_2$
*overwrite* the weights encoding $\mathcal{T}_1$:

$$
\theta^* = \arg\min_\theta \mathcal{L}_2(\theta)
\quad\Rightarrow\quad
\mathcal{L}_1(\theta^*) \gg \mathcal{L}_1(\theta_1^*)
$$

> **Historical note**: First formalised by McCloskey & Cohen (1989); studied empirically on
> Permuted MNIST by Goodfellow et al. (2013).


In [ ]:
# ============================================================
# Figure 1 – Conceptual illustration of catastrophic forgetting
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Catastrophic Forgetting: A Visual Introduction',
             fontsize=15, fontweight='bold', y=1.01)

# Panel A – Loss landscape
ax = axes[0]; ax.set_title('A. Loss Landscape')
θ = np.linspace(-3, 3, 300)
L1 = 0.5*(θ+1.5)**2; L2 = 0.5*(θ-1.5)**2
ax.plot(θ, L1, color='#60a5fa', lw=2.5, label='$\\mathcal{L}_1$ Task 1')
ax.plot(θ, L2, color='#f472b6', lw=2.5, label='$\\mathcal{L}_2$ Task 2')
ax.axvline(-1.5, color='#60a5fa', ls='--', alpha=0.5, label='$\\theta_1^*$')
ax.axvline( 1.5, color='#f472b6', ls='--', alpha=0.5, label='$\\theta_2^*$')
ax.annotate('Minimising $\\mathcal{L}_2$\nforgots Task 1',
            xy=(1.5,.05), xytext=(0,2.4),
            arrowprops=dict(arrowstyle='->', color='#fbbf24'), color='#fbbf24', fontsize=8)
ax.set_xlabel('Parameter θ'); ax.set_ylabel('Loss'); ax.legend(fontsize=8); ax.grid(True)

# Panel B – Accuracy over time
ax = axes[1]; ax.set_title('B. Accuracy vs Training Time')
ep = np.arange(20)
a1 = np.r_[np.linspace(0.1,0.97,10), np.linspace(0.97,0.12,10)]
a2 = np.r_[np.full(10,np.nan), np.linspace(0.1,0.95,10)]
ax.plot(ep, a1, color='#60a5fa', lw=2.5, label='Task 1 (trained first)')
ax.plot(ep, a2, color='#f472b6', lw=2.5, label='Task 2 (trained second)')
ax.axvline(10, color='#fbbf24', ls=':', lw=2, label='Task switch')
ax.fill_between(ep[10:], a1[10:], 0.97, alpha=0.15, color='#ef4444', label='Forgotten!')
ax.set_xlabel('Training Epoch'); ax.set_ylabel('Test Accuracy')
ax.set_ylim(0,1.05); ax.legend(fontsize=8); ax.grid(True)

# Panel C – Stability–Plasticity trade-off
ax = axes[2]; ax.set_title('C. Stability–Plasticity Dilemma')
s = np.linspace(0,1,100)
ax.plot(s, 1-s, color='#a78bfa', lw=3, label='Trade-off frontier')
ax.scatter([0.05],[0.95], color='#ef4444', s=140, zorder=5, label='Standard DNN (forgets!)')
ax.scatter([0.95],[0.05], color='#6b7280', s=140, zorder=5, label='Frozen model (rigid!)')
ax.scatter([0.70],[0.70], color='#10b981', s=180, zorder=5, marker='*', label='Ideal CL system')
ax.set_xlabel('Stability'); ax.set_ylabel('Plasticity')
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.legend(fontsize=8); ax.grid(True)

plt.tight_layout(); plt.show()


## 1.3  Biological Motivation

The brain is the ultimate continual learner. Key mechanisms that inspire CL algorithms:

| Biological Mechanism | Description | CL Inspiration |
|----------------------|-------------|----------------|
| **Hippocampal Replay** | Sleep replays recent experiences to neocortex | → Experience Replay |
| **Sparse Coding** | Only ~1–5% of neurons active per stimulus | → Parameter Isolation |
| **Synaptic Consolidation** | Important synapses tagged & protected (BCM rule) | → EWC, SI |
| **Complementary Learning Systems** | Hippocampus (fast) + Neocortex (slow) | → Dual-memory systems |
| **Predictive Coding** | Brain predicts inputs; only errors propagate | → PC networks |
| **Neuromodulation** | Dopamine gates memory consolidation | → Meta-learning |

### Complementary Learning Systems (CLS) Theory

McClelland et al. (1995):

$$
\underbrace{\text{Hippocampus}}_{\text{Fast, episodic}}
\xrightarrow{\text{sleep replay}}
\underbrace{\text{Neocortex}}_{\text{Slow, semantic}}
$$


## 1.4  Learning Paradigm Taxonomy

| Paradigm | Data access | Key goal | Forgetting? |
|----------|------------|----------|-------------|
| **Offline / Batch** | All at once | High accuracy | N/A |
| **Online** | One sample at a time | Fast adaptation | Possible |
| **Incremental** | New classes over time | Expand knowledge | Yes |
| **Lifelong** | Long task sequence | Knowledge accumulation | Yes |
| **Continual** | Sequential tasks, bounded memory | No forgetting | Central concern |

$$\boxed{\text{Continual Learning} \supset \text{Incremental} \supset \text{Online Learning}}$$


## 1.5  Real-World Applications

| Domain | Application | CL Challenge |
|--------|-------------|-------------|
| 🤖 **Robotics** | Learn new manipulation tasks | Retain old motor skills |
| 🚗 **Autonomous Driving** | Adapt to new cities / weather | Safe driving in old conditions |
| 🏥 **Medical AI** | Add rare disease detection | Not forget common diseases |
| 📱 **Recommendation** | Update taste models | Preserve long-term preferences |
| 🔤 **LLMs** | Continual pre-training / fine-tuning | Catastrophic forgetting of world knowledge |
| 👤 **Personal AI** | Learn user preferences privately | No cloud replay allowed |
| 📟 **Edge AI** | IoT sensor local learning | Extremely constrained memory |
| 🧬 **Neuroscience** | Model brain memory accumulation | Biological plausibility |


---
# PART 2 : Types of Continual Learning
---


## 2.1  Task-Incremental Learning (Task-IL)

**Definition**: Sequence of distinct tasks; task identity $t$ is provided at test time.

$$p(y \mid x, t) \quad t \in \{1,\ldots,T\} \text{ known at test time}$$

**Architecture**
```
Input x ──► Shared Backbone ──► Task-1 Head
                             ──► Task-2 Head
                             ──► ...
```

| ✅ Advantages | ❌ Limitations |
|--------------|---------------|
| Easiest CL setting | Task oracle required at test time |
| Clear task boundaries | Unrealistic in many deployments |

**Real examples**: Split MNIST (5 tasks: {0,1}, {2,3}, …, {8,9}), Split CIFAR-10.

---

## 2.2  Domain-Incremental Learning (Domain-IL)

**Definition**: Same task (output space), different input distributions. Task ID **not** available at test time.

$$p_t(y \mid x) \neq p_{t'}(y \mid x) \quad \text{but} \quad \mathcal{Y}_t = \mathcal{Y}_{t'} \;\forall t,t'$$

**Real examples**: Permuted MNIST (same 10 digits, different pixel layouts), Medical imaging across hospitals.

---

## 2.3  Class-Incremental Learning (Class-IL)

**Definition**: New classes added over time; model must distinguish **all classes seen so far** without task ID.

$$\mathcal{Y}_1 \subset \mathcal{Y}_2 \subset \cdots \subset \mathcal{Y}_T$$

**Hardest standard CL setting** — most replay-free methods fail dramatically here.

---

## 2.4  Online Continual Learning

Data arrives as a **continuous stream**; model updated one mini-batch at a time.

$$\theta \leftarrow \text{Update}(\theta, \mathbf{B}_s), \quad |\mathbf{B}_s| \ll |\mathcal{D}|$$

---

## 2.5  Streaming Continual Learning

Extreme case: model sees each sample **exactly once** (no epochs, no replay).

---

## 2.6  Few-Shot Continual Learning

New tasks arrive with $K \in \{1,5,10\}$ labeled examples.  
Combines few-shot learning and catastrophic forgetting challenges.

---

## 2.7  Test-Time Adaptation (TTA)

Model adapts at inference using only unlabeled test data:

$$\theta_t = \arg\min_\theta \mathcal{L}_{\text{unsupervised}}(\theta;\mathcal{D}_t^{\text{test}})$$


In [ ]:
# ============================================================
# Figure 2 – CL Settings Comparison Table (visual)
# ============================================================
fig, ax = plt.subplots(figsize=(16, 5))
ax.axis('off'); fig.patch.set_facecolor('#0d1117')

columns = ['Setting','Task ID\nat Train','Task ID\nat Test','Same\nOutput Space',
           'Boundaries','Data/Task','Difficulty','Benchmark']
rows = [
    ['Task-IL',   '✅','✅','❌','Clear', 'Many', '⭐☆☆☆☆','Split MNIST'],
    ['Domain-IL', '✅','❌','✅','Clear', 'Many', '⭐⭐⭐☆☆','Permuted MNIST'],
    ['Class-IL',  '✅','❌','❌','Clear', 'Many', '⭐⭐⭐⭐☆','Split CIFAR-100'],
    ['Online CL', '✅','❌','Mixed','Blurry','Mini-batch','⭐⭐⭐⭐☆','Online MNIST'],
    ['Streaming', '✅','❌','Mixed','None','1 sample','⭐⭐⭐⭐⭐','Stream bench.'],
    ['Few-Shot CL','✅','❌','❌','Clear','K shots','⭐⭐⭐⭐⭐','FSCIL bench.'],
    ['TTA',        '❌','❌','✅','None','0 labels','⭐⭐⭐☆☆','CIFAR-10-C'],
]
tbl = ax.table(cellText=rows, colLabels=columns, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.1, 2.2)
for (r,c), cell in tbl.get_celld().items():
    cell.set_edgecolor('#30363d')
    if r == 0:
        cell.set_facecolor('#21262d'); cell.set_text_props(color='#58a6ff', fontweight='bold')
    elif r % 2:
        cell.set_facecolor('#0d1117'); cell.set_text_props(color='#c9d1d9')
    else:
        cell.set_facecolor('#161b22'); cell.set_text_props(color='#c9d1d9')
ax.set_title('Continual Learning Settings – Comparison Table',
             color='#c9d1d9', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()


---
# PART 3 : Continual Learning Methods
---

All methods inherit from a common `ContinualLearningStrategy` base class:

```
ContinualLearningStrategy (ABC)
├── before_task(task_id, dataset)
├── train_epoch(loader, task_id) → float
├── after_task(task_id, dataset)
└── evaluate(loader) → float

Concrete subclasses:
├── NaiveStrategy          (fine-tuning baseline)
├── ExperienceReplayStrategy  (ring buffer)
├── ReservoirReplayStrategy   (reservoir sampling)
├── EWCStrategy            (Fisher-based regularization)
├── OnlineEWCStrategy      (consolidated Fisher)
├── SIStrategy             (path-integral importance)
├── LwFStrategy            (knowledge distillation)
├── AGEMStrategy           (gradient projection)
└── ProgressiveNNStrategy  (expanding architecture)
```


In [ ]:
# ============================================================
# PART 3 – Base Network & Abstract Strategy
# ============================================================

class MLP(nn.Module):
    '''Multi-Layer Perceptron for MNIST (784 -> hidden -> output).

    Args:
        input_dim  : Flattened input size (784 for MNIST).
        hidden_dims: Tuple of hidden layer widths.
        output_dim : Number of output classes.
        dropout    : Dropout probability (0 = disabled).
    '''
    def __init__(self, input_dim=784, hidden_dims=(256,256), output_dim=10, dropout=0.0):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        self.backbone   = nn.Sequential(*layers)
        self.classifier = nn.Linear(prev, output_dim)

    def forward(self, x):
        return self.classifier(self.backbone(x.view(x.size(0), -1)))

    def get_features(self, x):
        '''Return backbone (pre-classifier) activations.'''
        return self.backbone(x.view(x.size(0), -1))


class ContinualLearningStrategy(ABC):
    '''Abstract base class for all continual learning strategies.

    Subclasses **must** implement `train_epoch`.
    Optionally override `before_task` and `after_task`.
    '''

    def __init__(self, model: nn.Module, optimizer: optim.Optimizer,
                 device: torch.device, name: str = 'CL Strategy'):
        self.model     = model.to(device)
        self.optimizer = optimizer
        self.device    = device
        self.name      = name

    @abstractmethod
    def train_epoch(self, loader: DataLoader, task_id: int) -> float:
        '''Train one epoch. Returns mean loss.'''

    def before_task(self, task_id: int, dataset: Dataset) -> None:
        '''Called before training task `task_id` starts.'''

    def after_task(self, task_id: int, dataset: Dataset) -> None:
        '''Called after training task `task_id` completes.'''

    @torch.no_grad()
    def evaluate(self, loader: DataLoader) -> float:
        '''Compute accuracy on the given loader.'''
        self.model.eval()
        correct = total = 0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            correct += (self.model(x).argmax(1) == y).sum().item()
            total   += y.size(0)
        self.model.train()
        return correct / total if total else 0.0

    def __repr__(self):
        return f'{self.__class__.__name__}({self.name})'


# ─── Universal training loop ────────────────────────────────────
def run_continual_experiment(
        strategy: ContinualLearningStrategy,
        train_datasets: List[Dataset],
        test_datasets:  List[Dataset],
        epochs_per_task: int = 3,
        batch_size: int = 256,
        verbose: bool = True,
) -> np.ndarray:
    '''Run a full CL experiment.

    Returns:
        acc_matrix : shape (n_tasks*epochs_per_task + 1, n_tasks)
            acc_matrix[e, t] = accuracy on task t after global epoch e.
    '''
    n_tasks      = len(train_datasets)
    test_loaders = [DataLoader(ds, batch_size=512, shuffle=False) for ds in test_datasets]

    acc_matrix = [[strategy.evaluate(tl) for tl in test_loaders]]   # epoch 0 = untrained

    for task_id, train_ds in enumerate(train_datasets):
        strategy.before_task(task_id, train_ds)
        loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)

        for epoch in range(epochs_per_task):
            loss = strategy.train_epoch(loader, task_id)
            row  = [strategy.evaluate(tl) for tl in test_loaders]
            acc_matrix.append(row)
            if verbose:
                accs = ' | '.join(f'T{i}:{a:.3f}' for i, a in enumerate(row))
                print(f'  [{strategy.name}] Task {task_id} Ep {epoch+1}/{epochs_per_task} '
                      f'Loss:{loss:.4f} | {accs}')

        strategy.after_task(task_id, train_ds)

    return np.array(acc_matrix)


print('✅  Base classes defined.')


## 3.1  Naive Strategy (Catastrophic Forgetting Baseline)

Standard cross-entropy on the current task only — **no CL mechanism**.  
This is the lower bound every CL method should beat.

$$\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}_t(\theta)$$

No constraint from previous tasks → gradients freely overwrite old knowledge.


In [ ]:
# ============================================================
# 3.1 – Naive (fine-tuning) baseline
# ============================================================

class NaiveStrategy(ContinualLearningStrategy):
    '''Naive sequential fine-tuning — no continual learning.

    Catastrophic forgetting baseline.
    Uses cross-entropy on the current task data only.
    '''

    def train_epoch(self, loader: DataLoader, task_id: int) -> float:
        self.model.train()
        total = 0.0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            self.optimizer.zero_grad()
            loss = F.cross_entropy(self.model(x), y)
            loss.backward(); self.optimizer.step()
            total += loss.item()
        return total / len(loader)

print('✅  NaiveStrategy defined.')


## 3.2  Replay Methods

### Core Idea
Maintain a bounded **memory buffer** $\mathcal{M}$ of past examples.
When learning task $t$, replay samples from $\mathcal{M}$ alongside new data:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{new}}(\theta;\mathcal{D}_t) + \lambda\cdot\mathcal{L}_{\text{replay}}(\theta;\mathcal{M})$$

### Biological Motivation
During **sleep**, the hippocampus replays recent episodes to consolidate them in neocortex
(*Complementary Learning Systems* theory, McClelland et al., 1995).

### 3.2.1 Experience Replay – Ring Buffer

Simplest strategy: store a fixed-capacity buffer; evict random items when full.

**Pseudo-code**
```
buffer M ← {}
For each task t:
    For each batch B from D_t:
        B_replay = sample(M, |B|//2)
        loss = CE(model, B ∪ B_replay)
        update model with loss
    Add samples from B to M (capped at capacity)
```

### 3.2.2 Reservoir Sampling Buffer

Algorithm R (Vitter, 1985) ensures *every observed item has equal probability* of being in the buffer:

For item $i$ (0-indexed): keep with probability $\min(1, |M| / i)$; if kept, replace a random slot.

$$P(\text{item } i \in M) = \frac{|M|}{n_{\text{seen}}} \quad \forall i$$


In [ ]:
# ============================================================
# 3.2 – Replay Buffers & Strategies
# ============================================================

class ReplayBuffer:
    '''Fixed-capacity replay buffer (ring / random-eviction policy).

    Args:
        capacity: Maximum number of (x, y) pairs stored.
    '''
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.xs: list = []; self.ys: list = []

    def __len__(self): return len(self.xs)

    def add(self, x: torch.Tensor, y: torch.Tensor):
        for xi, yi in zip(x, y):
            if len(self.xs) < self.capacity:
                self.xs.append(xi.cpu()); self.ys.append(yi.cpu())
            else:
                idx = random.randint(0, self.capacity - 1)
                self.xs[idx] = xi.cpu(); self.ys[idx] = yi.cpu()

    def sample(self, n: int, device) -> Tuple[torch.Tensor, torch.Tensor]:
        n   = min(n, len(self.xs))
        idx = random.sample(range(len(self.xs)), n)
        return (torch.stack([self.xs[i] for i in idx]).to(device),
                torch.stack([self.ys[i] for i in idx]).to(device))


class ReservoirBuffer:
    '''Reservoir-sampling buffer – uniform sample over all seen data.

    Algorithm R (Vitter 1985):
        If buffer not full -> add item.
        Else -> keep with probability capacity / n_seen; replace random slot.
    '''
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.xs: list = []; self.ys: list = []
        self.n_seen   = 0

    def __len__(self): return len(self.xs)

    def add(self, x: torch.Tensor, y: torch.Tensor):
        for xi, yi in zip(x, y):
            self.n_seen += 1
            if len(self.xs) < self.capacity:
                self.xs.append(xi.cpu()); self.ys.append(yi.cpu())
            else:
                j = random.randint(0, self.n_seen - 1)
                if j < self.capacity:
                    self.xs[j] = xi.cpu(); self.ys[j] = yi.cpu()

    def sample(self, n: int, device) -> Tuple[torch.Tensor, torch.Tensor]:
        n   = min(n, len(self.xs))
        idx = random.sample(range(len(self.xs)), n)
        return (torch.stack([self.xs[i] for i in idx]).to(device),
                torch.stack([self.ys[i] for i in idx]).to(device))


class ExperienceReplayStrategy(ContinualLearningStrategy):
    '''Experience Replay with a random-eviction ring buffer.

    Each training step: combine current batch with a replay sample,
    then update the buffer with the current batch.

    Args:
        buffer_capacity  : Max buffer size.
        replay_batch_size: Number of replay samples per step.
    '''
    def __init__(self, model, optimizer, device, buffer_capacity=1000, replay_batch_size=64):
        super().__init__(model, optimizer, device, name='Experience Replay')
        self.buffer   = ReplayBuffer(buffer_capacity)
        self.replay_bs = replay_batch_size

    def train_epoch(self, loader, task_id) -> float:
        self.model.train(); total = 0.0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            if len(self.buffer) > 0:
                rx, ry = self.buffer.sample(self.replay_bs, self.device)
                x, y   = torch.cat([x, rx]), torch.cat([y, ry])
            self.optimizer.zero_grad()
            loss = F.cross_entropy(self.model(x), y)
            loss.backward(); self.optimizer.step()
            total += loss.item()
            self.buffer.add(x.detach()[:self.replay_bs], y.detach()[:self.replay_bs])
        return total / len(loader)


class ReservoirReplayStrategy(ContinualLearningStrategy):
    '''Replay with reservoir sampling (unbiased over full history).

    Algorithmically identical to ExperienceReplayStrategy but uses
    a reservoir buffer instead of a ring buffer.
    '''
    def __init__(self, model, optimizer, device, buffer_capacity=1000, replay_batch_size=64):
        super().__init__(model, optimizer, device, name='Reservoir Replay')
        self.buffer    = ReservoirBuffer(buffer_capacity)
        self.replay_bs = replay_batch_size

    def train_epoch(self, loader, task_id) -> float:
        self.model.train(); total = 0.0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            if len(self.buffer) > 0:
                rx, ry = self.buffer.sample(self.replay_bs, self.device)
                xc, yc = torch.cat([x, rx]), torch.cat([y, ry])
            else:
                xc, yc = x, y
            self.optimizer.zero_grad()
            loss = F.cross_entropy(self.model(xc), yc)
            loss.backward(); self.optimizer.step()
            total += loss.item()
            self.buffer.add(x.detach(), y.detach())
        return total / len(loader)

print('✅  Replay strategies defined.')


## 3.3  Elastic Weight Consolidation (EWC)

**Paper**: [Kirkpatrick et al., *Overcoming catastrophic forgetting in neural networks*, PNAS 2017](https://arxiv.org/abs/1612.00796).

### Derivation via Bayesian Inference

Start from the posterior on parameters given both tasks:

$$\log p(\theta\mid\mathcal{D}_A,\mathcal{D}_B)
  = \log p(\mathcal{D}_B\mid\theta)
  + \underbrace{\log p(\theta\mid\mathcal{D}_A)}_{\text{prior from task A}}
  - \text{const}$$

Approximate the task-A posterior as a **Gaussian** (Laplace approximation):

$$\log p(\theta\mid\mathcal{D}_A) \approx
  -\tfrac{1}{2}(\theta-\theta_A^*)^\top F_A\,(\theta-\theta_A^*)$$

where $F_A$ is the **Fisher Information Matrix**:

$$F_A = \mathbb{E}_{p(\mathcal{D}_A)}\!\left[
  \nabla_\theta\log p(y|x,\theta)\,\nabla_\theta\log p(y|x,\theta)^\top
\right]$$

Using the diagonal approximation $F_A \approx \operatorname{diag}(F_A)$:

$$\boxed{
  \mathcal{L}_{\text{EWC}} = \mathcal{L}_B(\theta)
  + \frac{\lambda}{2}\sum_i F_i\,(\theta_i - \theta_{A,i}^*)^2
}$$

### Intuition
- $F_i$ **large** → parameter $i$ important for old tasks → heavily penalise changes.
- $F_i$ **small** → parameter $i$ unimportant → allow free movement.

### Limitations
- One Fisher matrix per completed task → memory grows linearly with tasks.
- Diagonal approximation loses inter-parameter correlations.


In [ ]:
# ============================================================
# 3.3 – Elastic Weight Consolidation (EWC)
# ============================================================

class EWCStrategy(ContinualLearningStrategy):
    '''EWC (Kirkpatrick et al., PNAS 2017).

    Computes diagonal Fisher after each task and adds a quadratic
    penalty on important parameter changes for subsequent tasks.

    Args:
        ewc_lambda     : Regularisation strength λ.
        fisher_samples : Mini-batches used to estimate Fisher.
    '''

    def __init__(self, model, optimizer, device, ewc_lambda=400.0, fisher_samples=200):
        super().__init__(model, optimizer, device, name='EWC')
        self.ewc_lambda     = ewc_lambda
        self.fisher_samples = fisher_samples
        # List of (fisher_dict, theta_star_dict) per completed task
        self.ewc_data: List[Tuple[dict, dict]] = []

    # ── Fisher estimation ─────────────────────────────────────
    def _compute_fisher(self, dataset: Dataset) -> dict:
        '''Estimate diagonal Fisher: F_i = E[(∂ log p / ∂θ_i)²].'''
        self.model.eval()
        fisher = {n: torch.zeros_like(p) for n, p in self.model.named_parameters()}
        loader    = DataLoader(dataset, batch_size=64, shuffle=True)
        n_batches = max(1, self.fisher_samples // 64)
        for i, (x, y) in enumerate(loader):
            if i >= n_batches: break
            x, y = x.to(self.device), y.to(self.device)
            self.model.zero_grad()
            F.cross_entropy(self.model(x), y).backward()
            for n, p in self.model.named_parameters():
                if p.grad is not None:
                    fisher[n] += p.grad.data.pow(2)
        for n in fisher: fisher[n] /= n_batches
        self.model.train()
        return fisher

    # ── After-task hook ────────────────────────────────────────
    def after_task(self, task_id: int, dataset: Dataset) -> None:
        print(f'  [EWC] Computing Fisher for task {task_id}…')
        fisher = self._compute_fisher(dataset)
        params = {n: p.data.clone() for n, p in self.model.named_parameters()}
        self.ewc_data.append((fisher, params))

    # ── EWC penalty ────────────────────────────────────────────
    def _ewc_penalty(self) -> torch.Tensor:
        '''L_EWC = (λ/2) Σ_tasks Σ_i F_i (θ_i − θ*_i)²'''
        if not self.ewc_data:
            return torch.tensor(0.0, device=self.device)
        penalty = torch.tensor(0.0, device=self.device)
        for fisher, old_params in self.ewc_data:
            for n, p in self.model.named_parameters():
                if n in fisher:
                    penalty += (fisher[n].to(self.device) *
                                (p - old_params[n].to(self.device)).pow(2)).sum()
        return (self.ewc_lambda / 2) * penalty

    def train_epoch(self, loader, task_id) -> float:
        self.model.train(); total = 0.0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            self.optimizer.zero_grad()
            loss = F.cross_entropy(self.model(x), y) + self._ewc_penalty()
            loss.backward(); self.optimizer.step()
            total += loss.item()
        return total / len(loader)

print('✅  EWCStrategy defined.')


## 3.4  Online EWC

**Paper**: [Schwarz et al., *Progress & Compress*, ICML 2018](https://arxiv.org/abs/1805.06370).

EWC stores one Fisher per task → memory grows linearly.  
Online EWC maintains a **single running Fisher** via exponential moving average:

$$\tilde{F}^{(t)} = \gamma\,\tilde{F}^{(t-1)} + F^{(t)}$$

$$\mathcal{L}_{\text{OEWC}}
  = \mathcal{L}_t(\theta)
  + \frac{\lambda}{2}\sum_i \tilde{F}_i^{(t-1)}\,(\theta_i - \theta_i^{*(t-1)})^2$$

Memory: $\mathcal{O}(|\theta|)$ regardless of number of tasks.


In [ ]:
# ============================================================
# 3.4 – Online EWC
# ============================================================

class OnlineEWCStrategy(ContinualLearningStrategy):
    '''Online EWC (Schwarz et al., ICML 2018).

    Consolidates Fisher estimates across tasks into a single
    running mean — constant memory overhead O(|θ|).

    Args:
        ewc_lambda     : Regularisation strength.
        gamma          : EMA decay for Fisher consolidation.
        fisher_samples : Mini-batches for Fisher estimation.
    '''

    def __init__(self, model, optimizer, device, ewc_lambda=400.0,
                 gamma=1.0, fisher_samples=200):
        super().__init__(model, optimizer, device, name='Online EWC')
        self.ewc_lambda     = ewc_lambda
        self.gamma          = gamma
        self.fisher_samples = fisher_samples
        self.fisher_cons: Optional[dict] = None   # Consolidated Fisher
        self.params_ref:  Optional[dict] = None   # θ* at last consolidation

    def _compute_fisher(self, dataset):
        self.model.eval()
        fisher = {n: torch.zeros_like(p) for n, p in self.model.named_parameters()}
        loader    = DataLoader(dataset, batch_size=64, shuffle=True)
        n_batches = max(1, self.fisher_samples // 64)
        for i, (x, y) in enumerate(loader):
            if i >= n_batches: break
            x, y = x.to(self.device), y.to(self.device)
            self.model.zero_grad()
            F.cross_entropy(self.model(x), y).backward()
            for n, p in self.model.named_parameters():
                if p.grad is not None: fisher[n] += p.grad.data.pow(2)
        for n in fisher: fisher[n] /= n_batches
        self.model.train(); return fisher

    def after_task(self, task_id, dataset):
        new_f = self._compute_fisher(dataset)
        if self.fisher_cons is None:
            self.fisher_cons = new_f
        else:
            for n in self.fisher_cons:
                self.fisher_cons[n] = self.gamma * self.fisher_cons[n] + new_f[n]
        self.params_ref = {n: p.data.clone() for n, p in self.model.named_parameters()}

    def _penalty(self):
        if self.fisher_cons is None: return torch.tensor(0.0, device=self.device)
        penalty = torch.tensor(0.0, device=self.device)
        for n, p in self.model.named_parameters():
            if n in self.fisher_cons:
                diff = p - self.params_ref[n].to(self.device)
                penalty += (self.fisher_cons[n].to(self.device) * diff.pow(2)).sum()
        return (self.ewc_lambda / 2) * penalty

    def train_epoch(self, loader, task_id) -> float:
        self.model.train(); total = 0.0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            self.optimizer.zero_grad()
            (F.cross_entropy(self.model(x), y) + self._penalty()).backward()
            self.optimizer.step(); total += (F.cross_entropy(self.model(x), y) + self._penalty()).item()
        return total / len(loader)

print('✅  OnlineEWCStrategy defined.')


## 3.5  Synaptic Intelligence (SI)

**Paper**: [Zenke, Poole & Ganguli, *Continual Learning Through Synaptic Intelligence*, ICML 2017](https://arxiv.org/abs/1703.04200).

### Key Innovation
Compute importance **online during training** (not post-hoc like EWC) via the path integral
of each parameter's contribution to reducing the loss:

$$\omega_k^t = -\int_{t_0}^{t_1} \frac{\partial\mathcal{L}}{\partial\theta_k}\,\dot\theta_k\,dt$$

Normalised importance:

$$\Omega_k = \frac{\omega_k}{(\Delta_k)^2 + \xi}, \quad \Delta_k = \theta_k^{(t_1)} - \theta_k^{(t_0)}$$

**Regularisation:**

$$\mathcal{L}_{\text{SI}} = \mathcal{L}_t(\theta) + c\sum_k \Omega_k\,(\theta_k - \theta_k^*)^2$$

### Advantage over EWC
- No post-hoc Fisher computation needed.
- Captures causal contribution of each weight movement.


In [ ]:
# ============================================================
# 3.5 – Synaptic Intelligence (SI)
# ============================================================

class SIStrategy(ContinualLearningStrategy):
    '''Synaptic Intelligence (Zenke et al., ICML 2017).

    Accumulates online path-integral importance during training.
    After each task, updates consolidated importance Ω.

    Args:
        si_lambda: Regularisation strength.
        damping  : ξ — avoids division by zero in importance normalisation.
    '''

    def __init__(self, model, optimizer, device, si_lambda=1.0, damping=1e-3):
        super().__init__(model, optimizer, device, name='SI')
        self.si_lambda = si_lambda; self.damping = damping
        self._init_si()

    def _init_si(self):
        self.omega:     dict = {}   # Consolidated importance Ω_k
        self.W:         dict = {}   # Path-integral accumulator
        self.theta_old: dict = {}   # θ at start of current task
        for n, p in self.model.named_parameters():
            self.omega[n]     = torch.zeros_like(p.data)
            self.W[n]         = torch.zeros_like(p.data)
            self.theta_old[n] = p.data.clone()

    def before_task(self, task_id, dataset):
        '''Record θ at task start and reset accumulator W.'''
        for n, p in self.model.named_parameters():
            self.theta_old[n] = p.data.clone()
            self.W[n]         = torch.zeros_like(p.data)

    def after_task(self, task_id, dataset):
        '''Update Ω using accumulated W and Δθ.'''
        for n, p in self.model.named_parameters():
            delta           = p.data - self.theta_old[n]          # Δθ
            # Ω_k += W_k / (Δθ_k² + ξ)
            self.omega[n]  += self.W[n] / (delta.pow(2) + self.damping)
            self.omega[n]   = torch.clamp(self.omega[n], min=0)

    def _si_penalty(self):
        '''c · Σ_k Ω_k (θ_k − θ*_k)²'''
        pen = torch.tensor(0.0, device=self.device)
        for n, p in self.model.named_parameters():
            pen += (self.omega[n].to(self.device) *
                    (p - self.theta_old[n].to(self.device)).pow(2)).sum()
        return self.si_lambda * pen

    def train_epoch(self, loader, task_id) -> float:
        self.model.train(); total = 0.0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            self.optimizer.zero_grad()
            loss = F.cross_entropy(self.model(x), y) + self._si_penalty()
            loss.backward()
            # Accumulate W_k += -g_k · (θ_k − θ*_k)  before the update
            for n, p in self.model.named_parameters():
                if p.grad is not None:
                    self.W[n] -= p.grad.data * (p.data - self.theta_old[n])
            self.optimizer.step(); total += loss.item()
        return total / len(loader)

print('✅  SIStrategy defined.')


## 3.6  Learning without Forgetting (LwF)

**Paper**: [Li & Hoiem, *Learning without Forgetting*, ECCV 2016](https://arxiv.org/abs/1606.09282).

### Core Idea
Instead of storing old data, use the **old model's predictions** as soft targets
(knowledge distillation) on the new task data:

$$\mathcal{L}_{\text{LwF}}
  = \mathcal{L}_{\text{CE}}(f_\theta(x),y)
  + \lambda_{\text{old}}\cdot\mathcal{L}_{\text{KD}}(f_\theta(x), f_{\theta_0}(x))$$

### Temperature-Scaled Distillation Loss (Hinton et al., 2015)

$$\mathcal{L}_{\text{KD}}
  = -\sum_c\sigma\!\left(\frac{z_c^{\text{old}}}{T}\right)
    \log\sigma\!\left(\frac{z_c^{\text{new}}}{T}\right)
\cdot T^2$$

Higher temperature $T$ → softer distributions → more inter-class structure transferred.

### When LwF Struggles
- Large distribution shift between tasks.
- Many tasks (distillation error accumulates).


In [ ]:
# ============================================================
# 3.6 – Learning without Forgetting (LwF)
# ============================================================

class LwFStrategy(ContinualLearningStrategy):
    '''LwF (Li & Hoiem, ECCV 2016).

    Uses knowledge distillation from a frozen snapshot of the model
    as an implicit regulariser. No memory buffer required.

    Args:
        lwf_lambda  : Weight of the distillation loss.
        temperature : Softmax temperature T for distillation.
    '''

    def __init__(self, model, optimizer, device, lwf_lambda=1.0, temperature=2.0):
        super().__init__(model, optimizer, device, name='LwF')
        self.lwf_lambda  = lwf_lambda
        self.temperature = temperature
        self.old_model: Optional[nn.Module] = None

    def before_task(self, task_id, dataset):
        if task_id > 0:
            self.old_model = copy.deepcopy(self.model).eval()
            for p in self.old_model.parameters(): p.requires_grad_(False)

    def _kd_loss(self, logits_new, x):
        if self.old_model is None: return torch.tensor(0.0, device=self.device)
        T = self.temperature
        with torch.no_grad():
            logits_old = self.old_model(x)
        soft_old = F.softmax(logits_old / T, dim=1)
        log_new  = F.log_softmax(logits_new / T, dim=1)
        return F.kl_div(log_new, soft_old, reduction='batchmean') * (T ** 2)

    def train_epoch(self, loader, task_id) -> float:
        self.model.train(); total = 0.0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            self.optimizer.zero_grad()
            logits = self.model(x)
            loss   = F.cross_entropy(logits, y) + self.lwf_lambda * self._kd_loss(logits, x)
            loss.backward(); self.optimizer.step(); total += loss.item()
        return total / len(loader)

print('✅  LwFStrategy defined.')


## 3.7  A-GEM: Average Gradient Episodic Memory

**Papers**:  
- GEM: [Lopez-Paz & Ranzato, *Gradient Episodic Memory*, NeurIPS 2017](https://arxiv.org/abs/1706.08840).  
- A-GEM: [Chaudhry et al., *Efficient Lifelong Learning with A-GEM*, ICLR 2019](https://arxiv.org/abs/1812.00420).

### Core Idea (GEM)
Constrain the gradient update so it does **not increase** the episodic loss on any old task:

$$\min_g \tfrac{1}{2}\|g - g_t\|^2 \quad\text{s.t.}\quad \langle g, g_k\rangle \geq 0 \;\forall k<t$$

### A-GEM Simplification
Replace per-task constraint with a constraint against the **average reference gradient**:

$$g_{\text{ref}} = \frac{1}{|\mathcal{M}|}\sum_{k<t} g_k$$

**Projection** (applied when $\langle g_t, g_{\text{ref}}\rangle < 0$):

$$\tilde{g} = g_t - \frac{\langle g_t, g_{\text{ref}}\rangle}{\|g_{\text{ref}}\|^2}\,g_{\text{ref}}$$

Computational cost: $\mathcal{O}(|\theta|)$ per step — no QP solver needed.


In [ ]:
# ============================================================
# 3.7 – A-GEM
# ============================================================

class AGEMStrategy(ContinualLearningStrategy):
    '''A-GEM (Chaudhry et al., ICLR 2019).

    Projects gradient to satisfy the average episodic memory constraint
    in O(|θ|) — far cheaper than full GEM (quadratic program).

    Args:
        buffer_capacity : Total episodic memory budget.
        ref_batch_size  : Samples used to compute the reference gradient.
    '''

    def __init__(self, model, optimizer, device, buffer_capacity=500, ref_batch_size=64):
        super().__init__(model, optimizer, device, name='A-GEM')
        self.buffer   = ReservoirBuffer(buffer_capacity)
        self.ref_bs   = ref_batch_size

    def _reference_gradient(self) -> Optional[torch.Tensor]:
        if not len(self.buffer): return None
        rx, ry = self.buffer.sample(self.ref_bs, self.device)
        self.model.zero_grad()
        F.cross_entropy(self.model(rx), ry).backward()
        g = torch.cat([p.grad.data.clone().view(-1)
                       for p in self.model.parameters() if p.grad is not None])
        self.model.zero_grad(); return g

    @staticmethod
    def _project(g: torch.Tensor, g_ref: torch.Tensor) -> torch.Tensor:
        '''g̃ = g − (<g,g_ref>/||g_ref||²)·g_ref  if <g,g_ref><0 else g'''
        dot = torch.dot(g, g_ref)
        if dot >= 0: return g
        return g - (dot / (g_ref.dot(g_ref) + 1e-8)) * g_ref

    def train_epoch(self, loader, task_id) -> float:
        self.model.train(); total = 0.0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            self.optimizer.zero_grad()
            loss = F.cross_entropy(self.model(x), y)
            loss.backward()
            g_curr = torch.cat([p.grad.data.clone().view(-1)
                                 for p in self.model.parameters() if p.grad is not None])
            g_ref = self._reference_gradient()
            if g_ref is not None:
                g_proj = self._project(g_curr, g_ref)
                ofs = 0
                for p in self.model.parameters():
                    if p.grad is not None:
                        n = p.grad.numel()
                        p.grad.data.copy_(g_proj[ofs:ofs+n].view(p.shape)); ofs += n
            self.optimizer.step(); total += loss.item()
            self.buffer.add(x.detach(), y.detach())
        return total / len(loader)

print('✅  AGEMStrategy defined.')


## 3.8  Progressive Neural Networks

**Paper**: [Rusu et al., *Progressive Neural Networks*, arXiv 2016](https://arxiv.org/abs/1606.04671).

### Core Idea
**Never modify old weights.** Add a new **column** per task with **lateral connections**
from all previous (frozen) columns:

$$h_i^{(k)} = f\!\left(W_i^{(k)}h_{i-1}^{(k)} + \sum_{j<k}U_i^{(k:j)}h_{i-1}^{(j)}\right)$$

```
Task 1     Task 2          Task 3
  h₁¹        h₁²             h₁³
  ↕          ↕↖              ↕↖↖
  h₂¹ ──→   h₂²             h₂³
  ↕          ↕↖              ↕↖↖
 out¹       out²            out³
```

| ✅ Advantages | ❌ Limitations |
|--------------|---------------|
| **Zero forgetting** by design | Parameters grow $\mathcal{O}(T\cdot|\theta|)$ |
| Full forward transfer | Not practical for many tasks |


In [ ]:
# ============================================================
# 3.8 – Progressive Neural Networks (simplified flat version)
# ============================================================

class ProgressiveNNStrategy(ContinualLearningStrategy):
    '''Progressive Neural Networks (Rusu et al., 2016) – simplified.

    Each task gets its own MLP column.  Previous columns are frozen.
    Lateral connections are approximated by concatenating feature
    representations from all previous columns before the final layer.

    This simplified version avoids the recursive column coupling of
    the full algorithm while demonstrating the zero-forgetting property.
    '''

    def __init__(self, input_dim, hidden_dims, output_dim, device, lr=1e-3):
        dummy = nn.Linear(1, 1)
        super().__init__(dummy, optim.Adam(dummy.parameters()), device, name='Progressive NN')
        self.input_dim   = input_dim
        self.hidden_dims = hidden_dims
        self.output_dim  = output_dim
        self.lr          = lr
        self.columns:     List[nn.Module] = []
        self.col_opts:    List[optim.Optimizer] = []
        self.current_tid: int = 0

    def _make_column(self) -> nn.Module:
        layers, prev = [], self.input_dim
        for h in self.hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        layers.append(nn.Linear(prev, self.output_dim))
        return nn.Sequential(*layers).to(self.device)

    def before_task(self, task_id, dataset):
        # Freeze all existing columns
        for col in self.columns:
            for p in col.parameters(): p.requires_grad_(False)
        # New column
        col = self._make_column()
        self.columns.append(col)
        self.col_opts.append(optim.Adam(col.parameters(), lr=self.lr))
        self.current_tid = task_id
        total_p = sum(p.numel() for c in self.columns for p in c.parameters())
        print(f'  [PNN] Column {task_id} added. Total parameters: {total_p:,}')

    def train_epoch(self, loader, task_id) -> float:
        col = self.columns[task_id]
        col.train(); opt = self.col_opts[task_id]; total = 0.0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            opt.zero_grad()
            loss = F.cross_entropy(col(x.view(x.size(0),-1)), y)
            loss.backward(); opt.step(); total += loss.item()
        return total / len(loader)

    @torch.no_grad()
    def evaluate(self, loader, task_id=None) -> float:
        tid = task_id if task_id is not None else self.current_tid
        if tid >= len(self.columns): return 0.0
        col = self.columns[tid]; col.eval()
        correct = total = 0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            preds = col(x.view(x.size(0),-1)).argmax(1)
            correct += (preds == y).sum().item(); total += y.size(0)
        return correct / total if total else 0.0

print('✅  ProgressiveNNStrategy defined.')


## 3.9  Methods Summary

| Method | Family | Extra Memory | Forgetting | Transfer | Key Reference |
|--------|--------|-------------|------------|----------|---------------|
| Naive | Baseline | $O(1)$ | Very high | None | — |
| Experience Replay | Replay | $O(M)$ | Low-Med | Low | Ratcliff 1990 |
| Reservoir Replay | Replay | $O(M)$ | Low-Med | Low | Vitter 1985 |
| EWC | Regularisation | $O(2|\theta|T)$ | Medium | None | Kirkpatrick 2017 |
| Online EWC | Regularisation | $O(2|\theta|)$ | Medium | None | Schwarz 2018 |
| SI | Regularisation | $O(2|\theta|)$ | Medium | None | Zenke 2017 |
| LwF | Distillation | $O(2|\theta|)$ | Low-Med | High | Li 2016 |
| A-GEM | Gradient | $O(M)$ | Low | Medium | Chaudhry 2019 |
| Progressive NN | Architecture | $O(T|\theta|)$ | **Zero** | High | Rusu 2016 |


---
# PART 4 : Dataset Preparation
---


## 4.1  MNIST Benchmarks

| Benchmark | What varies | CL Setting | Tasks |
|-----------|------------|------------|-------|
| **Split MNIST** | Class subset | Task-IL / Class-IL | 5 (2 digits each) |
| **Permuted MNIST** | Pixel layout | Domain-IL | Any $N$ |
| **Rotated MNIST** | Orientation | Domain-IL | Any $N$ |

These benchmarks are the standard "test bed" of the CL field because they are fast,
well-understood, and probe different aspects of forgetting and transfer.


In [ ]:
# ============================================================
# PART 4 – Dataset utilities
# ============================================================

# ── Raw MNIST (normalised, flattened) ──────────────────────
def get_mnist(data_dir='./data'):
    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,),(0.3081,)),
        transforms.Lambda(lambda x: x.view(-1)),    # -> (784,)
    ])
    train = datasets.MNIST(data_dir, train=True,  download=True, transform=tfm)
    test  = datasets.MNIST(data_dir, train=False, download=True, transform=tfm)
    return train, test


# ── Split MNIST ────────────────────────────────────────────
class SplitMNIST(Dataset):
    '''MNIST restricted to specified digit classes.
    Labels are remapped to [0, len(classes)-1].
    '''
    def __init__(self, base: Dataset, classes: List[int], remap: bool=True):
        self.base      = base; self.remap = remap
        self.label_map = {c:i for i,c in enumerate(sorted(classes))}
        tgt = torch.tensor(base.targets)
        mask = sum(tgt == c for c in classes).bool()
        self.idx = mask.nonzero(as_tuple=True)[0].tolist()

    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        x, y = self.base[self.idx[i]]
        return x, (self.label_map[int(y)] if self.remap else int(y))


def make_split_mnist(train_ds, test_ds, n_tasks=5):
    cls = [(2*t, 2*t+1) for t in range(n_tasks)]
    return ([SplitMNIST(train_ds, list(c)) for c in cls],
            [SplitMNIST(test_ds,  list(c)) for c in cls],
            cls)


# ── Permuted MNIST ─────────────────────────────────────────
class PermutedMNIST(Dataset):
    '''MNIST with a fixed pixel permutation.  Task 0 = identity.'''
    def __init__(self, base, perm): self.base = base; self.perm = perm
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]; return x[self.perm], y

def make_permuted_mnist(train_ds, test_ds, n_tasks=5, seed=42):
    g = torch.Generator().manual_seed(seed)
    perms = [torch.arange(784)] + [torch.randperm(784, generator=g) for _ in range(n_tasks-1)]
    return ([PermutedMNIST(train_ds, p) for p in perms],
            [PermutedMNIST(test_ds,  p) for p in perms],
            perms)


# ── Rotated MNIST ──────────────────────────────────────────
class RotatedMNIST(Dataset):
    '''MNIST rotated by a fixed angle (degrees).'''
    def __init__(self, base, angle):
        self.base = base; self.angle = angle
        self.rot  = transforms.Compose([
            transforms.Lambda(lambda x: x.view(28,28).unsqueeze(0)),
            transforms.RandomRotation((angle, angle)),
            transforms.Lambda(lambda x: x.view(-1)),
        ])
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]; return self.rot(x), y

def make_rotated_mnist(train_ds, test_ds, n_tasks=5, max_angle=180):
    angles = np.linspace(0, max_angle, n_tasks)
    return ([RotatedMNIST(train_ds, a) for a in angles],
            [RotatedMNIST(test_ds,  a) for a in angles],
            angles)


# ── Download + build ───────────────────────────────────────
print('Downloading MNIST (may take a moment) …')
mnist_train_raw, mnist_test_raw = get_mnist()

N_TASKS = 5

split_train,    split_test,    split_cls   = make_split_mnist(mnist_train_raw, mnist_test_raw, N_TASKS)
permuted_train, permuted_test, perms       = make_permuted_mnist(mnist_train_raw, mnist_test_raw, N_TASKS)
rotated_train,  rotated_test,  rot_angles  = make_rotated_mnist(mnist_train_raw, mnist_test_raw, N_TASKS)

print(f'✅  Split MNIST    : {N_TASKS} tasks  {[list(c) for c in split_cls]}')
print(f'✅  Permuted MNIST : {N_TASKS} tasks  all 10 classes')
print(f'✅  Rotated MNIST  : {N_TASKS} tasks  angles {[f"{a:.0f}°" for a in rot_angles]}')


In [ ]:
# ============================================================
# PART 4 – Visualise all benchmarks
# ============================================================
fig, axes = plt.subplots(3, N_TASKS, figsize=(15, 9))
fig.suptitle('MNIST Continual Learning Benchmarks', fontsize=14, fontweight='bold', y=1.01)
row_labels = ['Split MNIST\n(Task-IL / Class-IL)',
              'Permuted MNIST\n(Domain-IL)',
              'Rotated MNIST\n(Domain-IL)']

def show(ax, ds, title, color):
    x, _ = ds[0]
    img  = x.numpy(); img = (img-img.min())/(img.max()-img.min()+1e-8)
    ax.imshow(img.reshape(28,28), cmap='gray')
    ax.set_title(title, color=color, fontsize=9); ax.axis('off')

for t in range(N_TASKS):
    c = COLORS['tasks'][t]
    show(axes[0,t], split_train[t],    f'Task {t}\n{list(split_cls[t])}', c)
    show(axes[1,t], permuted_train[t], f'Task {t}\nPerm {t}', c)
    show(axes[2,t], rotated_train[t],  f'Task {t}\n{rot_angles[t]:.0f}°', c)

for i, lbl in enumerate(row_labels):
    axes[i,0].set_ylabel(lbl, fontsize=8, color='#a3aab5',
                         rotation=0, labelpad=65, va='center')
plt.tight_layout(); plt.show()


---
# PART 5 : Evaluation Metrics
---

Let $A_{t,j}$ denote accuracy on **task $j$ after training on task $t$**.

## 5.1  Core CL Metrics

### Average Accuracy (AA)
$$\text{AA}_T = \frac{1}{T}\sum_{t=1}^T A_{T,t}$$
Mean accuracy over **all** tasks after completing all training.  *(Higher ↑ is better)*

---

### Forgetting (F)
$$F_T = \frac{1}{T-1}\sum_{t=1}^{T-1}\left(\max_{j\leq T} A_{j,t} - A_{T,t}\right)$$
Average drop from each task's **peak** accuracy to its **final** accuracy.  *(Lower ↓ is better)*

---

### Backward Transfer (BWT)
$$\text{BWT} = \frac{1}{T-1}\sum_{t=1}^{T-1}\left(A_{T,t} - A_{t,t}\right)$$
How much learning later tasks **affects** earlier tasks.  
$\text{BWT}<0$ = forgetting; $\text{BWT}>0$ = positive backward transfer.

---

### Forward Transfer (FWT)
$$\text{FWT} = \frac{1}{T-1}\sum_{t=2}^{T}\left(A_{t-1,t} - A_{0,t}\right)$$
How much earlier tasks **help** later tasks before they are trained.

---

### Plasticity
$$\text{Plasticity}_t = A_{t,t}$$
Accuracy right after training task $t$ — measures learning capacity.

---

### Stability
$$\text{Stability}_t = A_{T,t}$$
Final accuracy on task $t$ — measures retention.


In [ ]:
# ============================================================
# PART 5 – CLMetrics class
# ============================================================

class CLMetrics:
    '''Comprehensive CL evaluation metrics.

    Args:
        acc_matrix      : shape (n_epochs+1, n_tasks).
        epochs_per_task : Epochs trained per task.
        n_tasks         : Number of tasks.
    '''

    def __init__(self, acc_matrix: np.ndarray, epochs_per_task: int, n_tasks: int):
        self.R   = acc_matrix
        self.ept = epochs_per_task
        self.T   = n_tasks

        self.R_final = acc_matrix[-1]
        ends = [(t+1)*epochs_per_task for t in range(n_tasks)]
        self.R_after = np.array([acc_matrix[min(e, len(acc_matrix)-1), t]
                                  for t, e in enumerate(ends)])

    def average_accuracy(self) -> float:
        return float(np.mean(self.R_final))

    def forgetting(self) -> float:
        if self.T <= 1: return 0.0
        return float(np.mean([np.max(self.R[:,t]) - self.R_final[t]
                               for t in range(self.T-1)]))

    def backward_transfer(self) -> float:
        if self.T <= 1: return 0.0
        return float(np.mean(self.R_final[:self.T-1] - self.R_after[:self.T-1]))

    def forward_transfer(self, random_acc=None) -> float:
        if self.T <= 1: return 0.0
        rand  = random_acc if random_acc is not None else np.zeros(self.T)
        fwts  = []
        for t in range(1, self.T):
            e = t * self.ept
            before = self.R[min(e, len(self.R)-1), t] if e < len(self.R) else rand[t]
            fwts.append(before - rand[t])
        return float(np.mean(fwts))

    def plasticity(self) -> np.ndarray:  return self.R_after
    def stability(self)  -> np.ndarray:  return self.R_final

    def summary(self, name='') -> dict:
        return {
            'Name': name,
            'Avg Acc':  round(self.average_accuracy(), 4),
            'Forgetting': round(self.forgetting(), 4),
            'BWT':      round(self.backward_transfer(), 4),
            'FWT':      round(self.forward_transfer(), 4),
            'Plasticity': round(float(np.mean(self.plasticity())), 4),
            'Stability':  round(float(np.mean(self.stability())), 4),
        }


def print_metrics_table(results, epochs_per_task, n_tasks):
    rows = [CLMetrics(m, epochs_per_task, n_tasks).summary(name)
            for name, m in results.items()]
    keys = list(rows[0].keys())
    hdr  = ' | '.join(f'{k:>14}' for k in keys)
    print(hdr); print('─'*len(hdr))
    for r in rows:
        print(' | '.join(f'{str(v):>14}' for v in r.values()))

print('✅  CLMetrics defined.')


---
# PART 6 : Experiments
---

We benchmark six strategies on **Permuted MNIST** (Domain-IL, 5 tasks).

> ⏱️ **Estimated runtime**: ~5 min GPU / ~15–25 min CPU.


In [ ]:
# ============================================================
# PART 6 – Experiment configuration
# ============================================================
EPOCHS_PER_TASK = 3
BATCH_SIZE      = 256
LR              = 1e-3
INPUT_DIM       = 784
HIDDEN_DIMS     = (256, 256)
OUTPUT_DIM      = 10

def fresh(input_dim=INPUT_DIM, hidden_dims=HIDDEN_DIMS,
          output_dim=OUTPUT_DIM, lr=LR):
    '''Create a fresh (randomly-initialised) MLP + Adam optimizer.'''
    model = MLP(input_dim, hidden_dims, output_dim).to(DEVICE)
    return model, optim.Adam(model.parameters(), lr=lr)

print('Experiment configuration:')
print(f'  Tasks:       {N_TASKS}')
print(f'  Epochs/task: {EPOCHS_PER_TASK}')
print(f'  Batch size:  {BATCH_SIZE}')
print(f'  Arch:        MLP {INPUT_DIM}->{HIDDEN_DIMS}->{OUTPUT_DIM}')
print(f'  Benchmark:   Permuted MNIST (Domain-IL)')
print(f'  Device:      {DEVICE}')


In [ ]:
# ============================================================
# PART 6 – Run all experiments
# ============================================================
results = {}   # name -> accuracy matrix

def run(name, strategy):
    print(f'\n{"="*55}\n{name}\n{"="*55}')
    mat = run_continual_experiment(
        strategy, permuted_train, permuted_test,
        EPOCHS_PER_TASK, BATCH_SIZE, verbose=True)
    results[name] = mat
    return mat

# 1. Naive
m, opt = fresh(); run('Naive', NaiveStrategy(m, opt, DEVICE))

# 2. Experience Replay
m, opt = fresh(); run('Replay', ExperienceReplayStrategy(m, opt, DEVICE, 1000, 64))

# 3. EWC
m, opt = fresh(); run('EWC', EWCStrategy(m, opt, DEVICE, 400.0, 500))

# 4. SI
m, opt = fresh(); run('SI', SIStrategy(m, opt, DEVICE, 1.0, 1e-3))

# 5. LwF
m, opt = fresh(); run('LwF', LwFStrategy(m, opt, DEVICE, 1.0, 2.0))

# 6. A-GEM
m, opt = fresh(); run('A-GEM', AGEMStrategy(m, opt, DEVICE, 500, 64))

print('\n✅  All experiments complete!')


In [ ]:
# ============================================================
# PART 6 – Metrics summary table
# ============================================================
print('\n📊  CONTINUAL LEARNING METRICS (Permuted MNIST, 5 tasks)')
print('='*80)
print_metrics_table(results, EPOCHS_PER_TASK, N_TASKS)


---
# PART 7 : Visualization Suite
---


In [ ]:
# ============================================================
# PART 7.1 – Accuracy Heatmaps
# R[i,j] = accuracy on task j after completing training on task i
# ============================================================

def accuracy_heatmap(acc_matrix, name, ax, ept=EPOCHS_PER_TASK):
    snap = np.array([acc_matrix[(t+1)*ept] for t in range(acc_matrix.shape[1]-1)
                     if (t+1)*ept < len(acc_matrix)] +
                    ([acc_matrix[-1]] if acc_matrix.shape[1]*ept >= len(acc_matrix) else []))
    # Robust: rebuild as (n_tasks, n_tasks) snapshot matrix
    n = acc_matrix.shape[1]
    snap_mat = np.zeros((n,n))
    for t in range(n):
        e = min((t+1)*ept, len(acc_matrix)-1)
        snap_mat[t] = acc_matrix[e]
    cmap = LinearSegmentedColormap.from_list('',['#1a0a2e','#3b82f6','#10b981','#84cc16'])
    im = ax.imshow(snap_mat, cmap=cmap, vmin=0, vmax=1, aspect='auto')
    ax.set_title(name, fontsize=10)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels([f'T{i}' for i in range(n)], fontsize=8)
    ax.set_yticklabels([f'T{i}' for i in range(n)], fontsize=8)
    ax.set_xlabel('Evaluated on Task', fontsize=8)
    ax.set_ylabel('After Training Task', fontsize=8)
    for i in range(n):
        for j in range(n):
            clr = 'white' if snap_mat[i,j] > 0.5 else '#8b949e'
            ax.text(j, i, f'{snap_mat[i,j]:.2f}', ha='center', va='center',
                    fontsize=7, color=clr)
    return im

n_met = len(results)
cols  = 3; rows = (n_met + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
axes = axes.flatten()
fig.suptitle('Accuracy Heatmaps – R[i,j] = Acc on Task j After Training Task i',
             fontsize=13, fontweight='bold')

for k, (name, mat) in enumerate(results.items()):
    im = accuracy_heatmap(mat, name, axes[k])
for k in range(n_met, len(axes)): axes[k].axis('off')

plt.colorbar(im, ax=axes[:n_met], shrink=0.6, label='Accuracy')
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# PART 7.2 – Learning Curves & Forgetting Curves
# ============================================================

METHOD_COLORS = {
    'Naive': COLORS['naive'], 'Replay': COLORS['replay'],
    'EWC':   COLORS['ewc'],   'SI':     COLORS['si'],
    'LwF':   COLORS['lwf'],   'A-GEM':  COLORS['agem'],
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Learning Curves – All Methods on Permuted MNIST', fontsize=13, fontweight='bold')

n_ep = N_TASKS * EPOCHS_PER_TASK
x_ep = np.arange(n_ep + 1)

# Left: Average accuracy over all tasks
ax = axes[0]; ax.set_title('Average Accuracy (All Tasks)')
for name, mat in results.items():
    avg = [np.mean(mat[e]) for e in range(len(mat))]
    ax.plot(x_ep[:len(avg)], avg, color=METHOD_COLORS.get(name,'#fff'),
            lw=2.5, label=name, marker='o', ms=3)
for t in range(1, N_TASKS):
    ax.axvline(t*EPOCHS_PER_TASK, color='#30363d', ls='--', lw=1)
ax.set_xlabel('Epoch'); ax.set_ylabel('Avg Accuracy')
ax.set_ylim(0,1.05); ax.legend(fontsize=9); ax.grid(True)

# Right: Forgetting at end of each task
ax = axes[1]; ax.set_title('Average Forgetting After Each Task')
for name, mat in results.items():
    f_vals = []
    for t in range(1, N_TASKS):
        e = (t+1)*EPOCHS_PER_TASK
        m = CLMetrics(mat[:min(e+1, len(mat))], EPOCHS_PER_TASK, t+1)
        f_vals.append(m.forgetting())
    ax.plot(range(1, N_TASKS), f_vals, color=METHOD_COLORS.get(name,'#fff'),
            lw=2.5, label=name, marker='s', ms=5)
ax.set_xlabel('After Task #'); ax.set_ylabel('Forgetting ↓')
ax.set_xticks(range(1,N_TASKS))
ax.legend(fontsize=9); ax.grid(True)

plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# PART 7.3 – Final Metrics Bar Chart
# ============================================================

metrics_keys = ['Avg Acc', 'Forgetting', 'BWT']
summaries    = {name: CLMetrics(mat, EPOCHS_PER_TASK, N_TASKS).summary(name)
                for name, mat in results.items()}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Final CL Metrics Comparison', fontsize=13, fontweight='bold')

for ax, key in zip(axes, metrics_keys):
    names = list(summaries.keys())
    vals  = [summaries[n][key] for n in names]
    bar_colors = [METHOD_COLORS.get(n, '#a78bfa') for n in names]
    bars = ax.bar(names, vals, color=bar_colors, edgecolor='#30363d', lw=1.5, width=0.6)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(key); ax.set_xticklabels(names, rotation=30, ha='right')
    ax.grid(True, axis='y', alpha=0.4)
    if key == 'Avg Acc': ax.set_ylim(0,1.1)

plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# PART 7.4 – Feature Space (PCA) using the EWC model
# ============================================================

def plot_feature_pca(model, datasets, device, title='PCA Feature Space', n=300):
    model.eval(); feats, tasks = [], []
    with torch.no_grad():
        for tid, ds in enumerate(datasets):
            loader = DataLoader(ds, batch_size=n, shuffle=True)
            x, _   = next(iter(loader))
            f      = model.get_features(x.to(device)).cpu().numpy()
            feats.append(f[:n]); tasks.append(np.full(min(n,len(f)), tid))
    feats = np.concatenate(feats); tasks = np.concatenate(tasks)

    pca = PCA(n_components=2)
    emb = pca.fit_transform(feats)

    fig, ax = plt.subplots(figsize=(8, 6))
    for tid in range(len(datasets)):
        mask = tasks == tid
        ax.scatter(emb[mask,0], emb[mask,1], c=COLORS['tasks'][tid],
                   label=f'Task {tid}', alpha=0.6, s=20)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Re-run EWC to get a trained model we can inspect
m_ewc, opt_ewc = fresh()
ewc_strat = EWCStrategy(m_ewc, opt_ewc, DEVICE, 400.0, 200)
_ = run_continual_experiment(ewc_strat, permuted_train, permuted_test,
                              EPOCHS_PER_TASK, BATCH_SIZE, verbose=False)
plot_feature_pca(ewc_strat.model, permuted_test, DEVICE,
                 title='EWC – Feature Space (PCA) on Permuted MNIST')


In [ ]:
# ============================================================
# PART 7.5 – Weight Distribution Histograms
# ============================================================

def plot_weight_dist(model, title='Weight Distributions'):
    params = [(n, p.data.cpu().numpy().flatten())
              for n, p in model.named_parameters() if 'weight' in n]
    fig, axes = plt.subplots(1, len(params), figsize=(5*len(params), 4))
    if len(params)==1: axes=[axes]
    fig.suptitle(title, fontsize=12, fontweight='bold')
    for ax, (n, w) in zip(axes, params):
        ax.hist(w, bins=60, color='#3b82f6', alpha=0.8, edgecolor='#1e3a5f')
        ax.axvline(0, color='#ef4444', lw=1.5, ls='--')
        ax.set_title(n.replace('.weight',''), fontsize=8)
        ax.text(0.03, 0.96, f'μ={w.mean():.3f}\nσ={w.std():.3f}',
                transform=ax.transAxes, va='top', fontsize=8, color='#c9d1d9',
                bbox=dict(boxstyle='round',facecolor='#21262d',alpha=0.8))
        ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

plot_weight_dist(ewc_strat.model, 'EWC Model – Final Weight Distributions')


---
# PART 8 : Frequently Asked Questions
---

## Q1  Why does catastrophic forgetting happen?

Neural networks encode knowledge in weight values.  
Training on task B computes $\nabla_\theta\mathcal{L}_B$ and updates *all* weights.  
Since there is no mechanism to protect task A's weights, they are overwritten.

$$\theta^*_B \neq \theta^*_A \;\Rightarrow\; \mathcal{L}_A(\theta^*_B) \gg \mathcal{L}_A(\theta^*_A)$$

---

## Q2  Why does replay work?

Replay ensures the gradient is a *mixture* of old and new task signals:

$$g_{\text{combined}} = \nabla\mathcal{L}_{\text{new}} + \lambda\,\nabla\mathcal{L}_{\text{replay}}$$

This prevents the gradient from pointing purely toward task B's minimum —
it is pulled back toward a region that satisfies task A's constraints too.

---

## Q3  Why does EWC work?

EWC identifies which weights are *important* for old tasks (via the Fisher Information Matrix)
and adds a quadratic penalty that prevents those weights from moving:

$$\mathcal{L}_{\text{EWC}} = \mathcal{L}_{\text{new}} + \tfrac{\lambda}{2}\sum_i F_i(\theta_i - \theta_i^*)^2$$

High $F_i$ → heavily penalise change → parameter protected.

---

## Q4  Replay vs Regularisation — key differences?

| Aspect | Replay | Regularisation |
|--------|--------|----------------|
| Mechanism | Stores / generates old data | Modifies loss function |
| Memory | $O(M)$ raw samples | $O(2|\theta|)$ |
| Privacy | Stores raw data ⚠️ | No raw data stored ✅ |
| Scales with tasks | Buffer dilutes older data | Penalty accumulates |

---

## Q5  Can continual learning replace offline training?

Not yet for most high-stakes applications.  
Offline training (full data access) almost always outperforms CL given enough compute.  
CL is motivated by **practical constraints**: privacy, storage limits, streaming data.

---

## Q6  Online learning vs Continual learning?

| | Online Learning | Continual Learning |
|--|-----------------|-------------------|
| Primary goal | Fast single-sample updates | Retaining multiple tasks |
| Task structure | No explicit task change | Explicit task sequences |
| Forgetting concern | Not primary | Central concern |

---

## Q7  Common CL benchmark datasets?

| Dataset | Setting | Complexity |
|---------|---------|------------|
| Split / Permuted / Rotated MNIST | Task-IL / Domain-IL | Low |
| Split CIFAR-10/100 | Task-IL, Class-IL | Medium |
| Tiny ImageNet | Class-IL | Medium-High |
| CORe50 | Object recognition | High |
| CLiMB (NLP) | Language model CL | Emerging |


---
# PART 9 : Mini Research Section
---


## 9.1  Current Challenges

| Challenge | Status |
|-----------|--------|
| Stability–Plasticity trade-off | No decisive solution yet |
| Task-free / blurry-boundary CL | Partial solutions (DER++, GSS) |
| Class-IL without replay | Largely unsolved |
| LLM-scale CL | Emerging — CPT, LFPT5 |
| Theoretical guarantees | Very few — PAC-Bayes bounds just appearing |
| Biologically plausible CL | Early stage — PC networks, STDP |

---

## 9.2  Predictive Coding and Continual Learning

### What is Predictive Coding (PC)?

PC proposes that the brain **minimises prediction errors** at every level of the cortical hierarchy:

$$\varepsilon_l = x_l - f_l(x_{l+1})$$

Learning minimises the **variational free energy**:

$$\mathcal{F} = \sum_l \tfrac{1}{2}\|\varepsilon_l\|^2$$

### Prospective Configuration (Song et al., *Nature Neuroscience* 2024)

The key PC training loop is:

$$\underbrace{x_l^* = \arg\min_{x_l}\mathcal{F}}_{\text{Inference (fast)}}
\quad\longrightarrow\quad
\underbrace{\Delta W_l \propto -\nabla_{W_l}\mathcal{F}\big|_{x=x^*}}_{\text{Learning (slow)}}$$

Because latent activities settle *before* weights update, the weight gradient is **smaller
and better targeted** — interfering less with previously learned mappings.

### Why PC May Help with CL

| PC Property | CL Benefit |
|-------------|-----------|
| Local learning rules | Hebbian-like — no global gradient required |
| Smaller weight updates | Less interference with old tasks |
| Inference phase separates perception from learning | Reduced task interference |
| Hierarchical representations | Natural multi-level task abstraction |

---

## 9.3  Future Research Directions

1. **PC + Replay** — combine prospective configuration with a small episodic buffer.
2. **Dendritic compartmentalisation** — neuron-level task separation via dendritic computation.
3. **Neuromodulatory gating** — use context signals (like ACh/dopamine) to switch between learn/retrieve modes.
4. **Sparse CL** — exploit sparsity to minimise parameter overlap between tasks.
5. **Federated CL** — maintain privacy while learning from distributed data streams.
6. **LLM continual fine-tuning** — efficient parameter-efficient methods (LoRA + CL).
7. **Spike-timing CL** — STDP-based CL for neuromorphic hardware.

---

## 9.4  Key Papers to Read

### Foundational
- [McCloskey & Cohen (1989). *Catastrophic interference in connectionist networks.*](https://doi.org/10.1016/S0166-4115(08)62336-9)
- [Goodfellow et al. (2013). *An empirical investigation of catastrophic forgetting.*](https://arxiv.org/abs/1312.6211)

### Regularisation
- [Kirkpatrick et al. (2017). *Overcoming catastrophic forgetting.*](https://arxiv.org/abs/1612.00796)
- [Zenke et al. (2017). *Continual learning through synaptic intelligence.*](https://arxiv.org/abs/1703.04200)

### Replay
- [Rolnick et al. (2019). *Experience replay for continual learning.*](https://arxiv.org/abs/1811.11682)
- [Buzzega et al. (2020). *Dark experience for general continual learning.*](https://arxiv.org/abs/2004.07211)

### Architecture
- [Rusu et al. (2016). *Progressive neural networks.*](https://arxiv.org/abs/1606.04671)

### Gradient
- [Chaudhry et al. (2019). *Efficient lifelong learning with A-GEM.*](https://arxiv.org/abs/1812.00420)

### Distillation
- [Li & Hoiem (2016). *Learning without forgetting.*](https://arxiv.org/abs/1606.09282)

### Biological / PC
- [Song et al. (2024). *Inferring neural activity before plasticity.* **Nature Neuroscience**.](https://doi.org/10.1038/s41593-023-01514-1)
- [Rao & Ballard (1999). *Predictive coding in the visual cortex.* **Nature Neuroscience**.](https://doi.org/10.1038/7206)

### Tools
- [Lomonaco et al. (2021). *Avalanche: An end-to-end library for CL.*](https://arxiv.org/abs/2104.00405)
- [Buzzega et al. (2021). *Mammoth: An extendable toolbox.*](https://github.com/aimagelab/mammoth)


In [ ]:
# ============================================================
# PART 9 – Research Landscape Visualisation
# ============================================================
fig, ax = plt.subplots(figsize=(13, 8))
ax.set_xlim(0,13); ax.set_ylim(0,8); ax.axis('off')
fig.suptitle('Continual Learning – Research Landscape & Future Directions',
             fontsize=13, fontweight='bold')

# Centre
cx, cy = 6.5, 4.0
ax.add_patch(plt.Circle((cx,cy), 1.15, color='#1f6feb', zorder=5))
ax.text(cx, cy, 'Continual\nLearning', ha='center', va='center',
        fontsize=11, fontweight='bold', color='white', zorder=6)

branches = [
    ((1.5,6.5),'Biological\nPlausibility','#ec4899'),
    ((11.5,6.5),'LLM\nFine-tuning','#8b5cf6'),
    ((1.5,1.5),'Neuromorphic\nHardware','#06b6d4'),
    ((11.5,1.5),'Federated\nLearning','#f59e0b'),
    ((6.5,7.5),'Generative\nReplay','#10b981'),
    ((6.5,0.5),'Meta-Learning\nfor CL','#ef4444'),
    ((0.3,4.0),'Predictive\nCoding','#3b82f6'),
    ((12.7,4.0),'Prompt-based\nCL','#84cc16'),
]
for (px,py), lbl, col in branches:
    dx,dy = cx-px, cy-py; d=max((dx**2+dy**2)**0.5,0.01)
    ndx,ndy = dx/d, dy/d
    ax.annotate('',xy=(px+ndx*0.55,py+ndy*0.55),
                xytext=(cx-ndx*1.2,cy-ndy*1.2),
                arrowprops=dict(arrowstyle='<->',color=col,lw=1.8))
    ax.add_patch(mpatches.FancyBboxPatch((px-0.85,py-0.45),1.7,0.9,
        boxstyle='round,pad=0.1',facecolor=col+'30',edgecolor=col,lw=2))
    ax.text(px,py,lbl,ha='center',va='center',fontsize=8.5,
            color=col,fontweight='bold')

plt.tight_layout(); plt.show()


---
# Summary & Conclusions
---

| Part | Key Takeaway |
|------|-------------|
| 1. Introduction | Catastrophic forgetting is the core challenge; brains solve it with replay, sparse coding & local learning |
| 2. Types | Task-IL → Domain-IL → Class-IL is a spectrum of increasing difficulty |
| 3. Methods | Replay (data-centric), Regularisation (parameter-centric), Architecture (capacity-centric) are the three families |
| 4. Datasets | Split / Permuted / Rotated MNIST test different CL properties |
| 5. Metrics | AA, Forgetting, BWT, FWT together give a complete picture |
| 6. Experiments | No single method dominates — trade-offs in memory, compute, and performance |
| 7. Visualisation | Heatmaps, learning curves, PCA are essential diagnostic tools |
| 8. FAQ | Replay prevents gradient interference; EWC protects important weights; they are complementary |
| 9. Research | Predictive coding offers biological stability; field evolves rapidly toward LLM-scale CL |

## Next Steps

1. 🔬 **Extend**: Run on Split CIFAR-10, compare Class-IL vs Task-IL.
2. 🧪 **Tune**: EWC λ, buffer sizes — sensitivity analysis.
3. 📚 **Read**: Start with Kirkpatrick et al. (2017) and Chaudhry et al. (2019).
4. 🛠️ **Frameworks**: [Avalanche](https://avalanche.continualai.org/) or [Mammoth](https://github.com/aimagelab/mammoth).
5. 🧠 **Predictive Coding**: Explore `continual_learning_pc.ipynb` in this repository.

---
*Notebook built for the NeuroMatch NeuroAI Project.*  
*Covers foundational-to-research-level continual learning in PyTorch.*
